## **IMPORTS**

In [11]:
# Apache Spark API
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import when, col, desc, count, round, isnan, sum, concat, lit, mean, std, monotonically_increasing_id

# Data Handling
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt

In [12]:
# Iniciando Spark Session
spark = SparkSession.builder \
    .appName("BigData_AC2") \
    .master("local[*]") \
    .config("spark.executor.memory", "12g") \
    .config("spark.driver.memory",   "12g") \
    .getOrCreate()

In [13]:
# Definição de Schema dos dados
df = spark.read.schema(
    StructType([
        StructField(name="Year", dataType=LongType(), nullable=True),
        StructField(name="Month", dataType=LongType(), nullable=True),
        StructField(name="DayofMonth", dataType=LongType(), nullable=True),
        StructField(name="DayOfWeek", dataType=LongType(), nullable=True),
        StructField(name="DepTime", dataType=DoubleType(), nullable=True),
        StructField(name="CRSDepTime", dataType=LongType(), nullable=True),
        StructField(name="ArrTime", dataType=DoubleType(), nullable=True),
        StructField(name="CRSArrTime", dataType=LongType(), nullable=True),
        StructField(name="UniqueCarrier", dataType=StringType(), nullable=True),
        StructField(name="FlightNum", dataType=LongType(), nullable=True),
        StructField(name="TailNum", dataType=StringType(), nullable=True),
        StructField(name="ActualElapsedTime", dataType=DoubleType(), nullable=True),
        StructField(name="CRSElapsedTime", dataType=DoubleType(), nullable=True),
        StructField(name="AirTime", dataType=DoubleType(), nullable=True),
        StructField(name="ArrDelay", dataType=DoubleType(), nullable=True),
        StructField(name="DepDelay", dataType=DoubleType(), nullable=True),
        StructField(name="Origin", dataType=StringType(), nullable=True),
        StructField(name="Dest", dataType=StringType(), nullable=True),
        StructField(name="Distance", dataType=DoubleType(), nullable=True),
        StructField(name="TaxiIn", dataType=DoubleType(), nullable=True),
        StructField(name="TaxiOut", dataType=DoubleType(), nullable=True),
        StructField(name="Cancelled", dataType=LongType(), nullable=True),
        StructField(name="isDelay", dataType=IntegerType(), nullable=True),
    ])
).parquet("../../../data/airline_clean.parquet", header=True)
print(f"N° samples: {df.count()}")

N° samples: 74008262


## **DATA CLEANING**

In [14]:
# Removendo colunas que não são viáveis aplicar OneHotEnconding
# São colunas com alto número de valores categóricos
# Aplicar OneHotEncoding nessas colunas, vai deixar o dataset esparso
df = df.drop(*['Dest', 'Origin', 'TailNum', 'FlightNum', 'Cancelled', 'DayofMonth', 'UniqueCarrier', 'DayOfWeek', 'Year'])

### MISSING DATA

In [15]:
# Criando tabela para análise de valores faltantes
missing = df.select([
    sum(
        when( col(c).isNull() | isnan(col(c)), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

In [16]:
# Aplicando transposição na tebela para orientar as colunas para linhas, e melhorar a visualização
missing = missing.selectExpr(
        "stack({0}, {1}) as (column, missing)".format(
            len(df.columns),
            ", ".join("'{}', {}".format(c, c) for c in df.columns)
        )
    ).withColumn(
        "porcentage",
        round(col("missing") / df.count() * 100, 2)
    ).orderBy(desc("missing"))

In [17]:
missing.show(truncate=False)

+-----------------+-------+----------+
|column           |missing|porcentage|
+-----------------+-------+----------+
|Month            |0      |0.0       |
|DepTime          |0      |0.0       |
|CRSDepTime       |0      |0.0       |
|ArrTime          |0      |0.0       |
|CRSArrTime       |0      |0.0       |
|ActualElapsedTime|0      |0.0       |
|CRSElapsedTime   |0      |0.0       |
|AirTime          |0      |0.0       |
|ArrDelay         |0      |0.0       |
|DepDelay         |0      |0.0       |
|Distance         |0      |0.0       |
|TaxiIn           |0      |0.0       |
|TaxiOut          |0      |0.0       |
|isDelay          |0      |0.0       |
+-----------------+-------+----------+



## **PRE-PROCESSING**

### SAMPLING DATA

In [18]:
train, test = df.randomSplit([0.8, 0.2], seed=2025)

In [19]:
display(train.count())
display(test.count())

59205249

14803013

### BALANCING TARGET COLUMN

In [20]:
# Visualizando a proporção do desbalanceamento entre as classes da coluna alvo
(train.groupBy('IsDelay').agg(
    count("IsDelay").alias("count"),
    (count("IsDelay") / train.count()).alias("percentage")
).show())

+-------+--------+------------------+
|IsDelay|   count|        percentage|
+-------+--------+------------------+
|      1|23436580|0.3958530771486157|
|      0|35768669|0.6041469228513844|
+-------+--------+------------------+



In [24]:
# Cálculo usando Regra de 3, para achar a porcentagem para aplicar o undersampling
(100*12332089)/35768669 - 100

-65.5226505632625

In [25]:
# Aplicando undersampling na classe majoritária
train = train.filter(
    col('isDelay') == 0
).sample(
    withReplacement=False, fraction=0.655, seed=2025
).unionByName(train.filter(col('isDelay') == 1))

In [26]:
# Verificando 
(train.groupBy('IsDelay').agg(
    count("IsDelay").alias("count"),
    (count("IsDelay") / train.count()).alias("percentage")
).show())

+-------+--------+------------------+
|IsDelay|   count|        percentage|
+-------+--------+------------------+
|      0|23432665|0.4999582348723561|
|      1|23436580| 0.500041765127644|
+-------+--------+------------------+



### NORMALIZATION WITH Z-SCORE

In [27]:
# Definindo colunas numéricas
columns_to_normalize = [
 'DepTime',
 'CRSDepTime',
 'ArrTime',
 'CRSArrTime',
 'ActualElapsedTime',
 'CRSElapsedTime',
 'AirTime',
 'ArrDelay',
 'DepDelay',
 'Distance',
 'TaxiIn',
 'TaxiOut'
]

# Calculando média e desvio padrão do set de treino, para cálculo do Z-Score
# Necessário usar média e desvio padrão do set de treino, para normalizar o set de teste
stats = (
    train.agg(
        *([mean(column).alias(f"{column}_mean") for column in columns_to_normalize] +
         [std(column).alias(f"{column}_std")  for column in columns_to_normalize])
    ).first().asDict()
)

In [28]:
# Z-score = (x - média) / desvio padrão
# Aplicando fórmula do Z-score no set de treino
for column in columns_to_normalize:
    train = train.withColumn(
        column,
        (col(column) - stats[f"{column}_mean"]) / stats[f"{column}_std"]
    )

# Aplicando fórmula do Z-score no set de teste
for column in columns_to_normalize:
    test = test.withColumn(
        column,
        (col(column) - stats[f"{column}_mean"]) / stats[f"{column}_std"]
    )

### ONE HOT ENCONDING

In [29]:
# Transformando coluna para tipo string
train = train.withColumn("Month", col("Month").cast(StringType()))
# Adicionando prefixo para colunas do one hot encoding
train = train.withColumn("Month", concat(lit('Month_'), col("Month")))

# Transformando coluna para tipo string
test = test.withColumn("Month", col("Month").cast(StringType()))
# Adicionando prefixo para colunas do one hot encoding
test = test.withColumn("Month", concat(lit('Month_'), col("Month")))

In [30]:
# One hot encoding coluna Month do treino
train = train.groupBy(
    train.drop('Month').columns
).pivot('Month').agg(
    count('*')
).fillna(0)

# One hot encoding coluna Month do teste
test = test.groupBy(
    test.drop('Month').columns
).pivot('Month').agg(
    count('*')
).fillna(0)

## **PERSIST DATA**

In [31]:
try:
    train.repartition(1).write.mode('overwrite').parquet('../../../data/train.parquet')
except Exception as e:
    print(e)

In [32]:
try:
    test.repartition(1).write.mode('overwrite').parquet('../../../data/test.parquet')
except Exception as e:
    print(e)